# 5. Charts

Turns each final long file into a folder of charts - dark, print-ready, one SVG
and one PNG per figure.

```
COMPENDIUM-ARAB SOCIETY\merged_long_files\<Chapter>_EN.xlsx
        ->  COMPENDIUM-ARAB SOCIETY\<chapter>_charts\
```

**Run this after notebook 3.** It reads the same long files notebook 4 does, so
it picks up the merged questionnaire rows and the calculated indicators. It is
independent of notebook 4 - neither needs the other.

## What comes out

**`<chapter>_charts.xlsx`** is the deliverable: one sheet per chart, each holding
that chart's picture with exactly the rows that produced it underneath, plus an
`Index` sheet linking to them all. The SVG and PNG of every chart sit in the same
folder.

Capturing the data inside the drawing primitives rather than in the chart
functions is deliberate - the primitive is the last place that still holds
exactly what was drawn, after every filter and guard has run, so a sheet cannot
drift away from the picture above it.

Excel places images through Pillow, which does not rasterise SVG, so **the
picture embedded in a sheet is the PNG**. The SVG beside it is the vector copy,
and the one to use anywhere that wants live text.

**Population** gets the numbered set the compendium prints - `1.1_pop_growth`,
`1.2_pop_size`, `1.3_sex_comp_gcc`, `1.6_sex_ratio`, `1.7_pop_age_sex`,
`1.8_fertility`, `1.9_life_exp`, `1.10_infant_mort`, `1.11_intl_migrant_gcc`,
`1.12_intl_migrant`, `1.13_refugees`, and one population pyramid per country.
The numbering keeps its gaps: no 1.4 or 1.5 was supplied to copy, so those
numbers are left free rather than closed up, and the files still line up with
the figure numbers in the report.

**Every other chapter** is charted from what its indicators actually carry,
three shapes per indicator where the data supports them:

| suffix | shape | drawn when |
|---|---|---|
| `_trend` | one line per country | the indicator has a country total over time |
| `_by_sex` | a panel per country, men against women | it is reported by sex |
| `_<breakdown>` | stacked shares, latest year | it splits into parts that sum to a whole |

`CHAPTERS` at the top of the config cell picks which chapters run; it is set to
`["Population"]`. Set it to `None` to chart every chapter found on disk.

Each folder also gets **`charts_index.csv`** - what every file shows - and
**`chart_data_findings.txt`**, below.

## The findings file

The charts refuse a figure rather than draw a wrong one, and every refusal is
written down. These are findings about the questionnaires, not about the code,
so they outlive the run and can go back to the country that reported them. What
gets caught:

- a value that is not a number (`'51.2+1.2'`, `'1345(الاعداد بالالف)'`);
- a figure orders of magnitude off its own country's series - Morocco's 2024
  population filed as `36,491`, in thousands, and Lebanon's 2022 as `100`;
- men and women that do not add up to their own reported total - Kuwait 2020's
  total is missing its leading digit, Tunisia 2015's male figure has one too
  many. Nothing in the file says which of the two is sound, so the country-year
  is dropped whole rather than guessed at;
- a total that had to be added up from its parts, and from which parts.

## Design notes

**The full rules live in `charts_design.md`, beside this notebook.** Read that
before changing anything about how a chart looks - it records what was measured,
what was chosen, and why, so none of it has to be worked out twice.

The short version:

- **The theme is the compendium's own**: white ground, black text, Times New
  Roman, the navy/pink/amber triad, and the country colours read back out of the
  published SVGs. Nothing here is invented styling.
- **A country keeps its colour across every chart**, because the lookup is on the
  name as it appears in the data. The published charts looked it up by printed
  label, so a legend that wrote `SaudiArabia` without the space missed and fell
  through to a default sequence - which is why Saudi Arabia is olive on one
  published figure and blue on the next.
- **Every size is measured, not guessed.** Figure heights, label margins and
  legend columns come from the rendered width of the text that has to fit, which
  is why nothing clips at any label length.
- **Titles are on by default.** The published figures carry none - the caption
  lives in the Word document - which works in the report and badly in a folder of
  sixty files. Set `DRAW_TITLES = False` in the primitives cell for the printed
  version.

In [ ]:
"""
CELL: Imports and logging setup.
"""
import logging
import re
import textwrap
from pathlib import Path

import matplotlib
matplotlib.use("Agg")          # write files; never try to open a window
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")

## Config / paths, and the theme

In [ ]:
"""
CELL: Configuration - paths, the dark chart theme, and the validated palettes.
"""
COMPENDIUM_PATH = Path(r"C:\Users\raffi\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY")

# Notebook 1 writes both languages into this one folder; the _EN / _AR suffix is
# already in each filename.
LONG_FILES_PATH = COMPENDIUM_PATH / "merged_long_files"

# One folder per chapter, beside the long files: Population -> population_charts.
CHARTS_ROOT = COMPENDIUM_PATH

# Leave CHAPTERS as None to chart every chapter found on disk. Set an explicit
# list to restrict one run, e.g. CHAPTERS = ["Population"].
CHAPTERS = ["Population"]
LANGUAGE = "EN"

# SVG keeps the text as text and scales without softening. PNG is written
# alongside because Word will not place an SVG that carries live text.
FORMATS = ["svg", "png"]

FINDINGS_NAME = "chart_data_findings.txt"

# ------------------------------------------------------------------ the theme
#
# The published compendium figures are light: a white ground, black text, Times
# New Roman, and no gridlines at all. Every colour here was read back out of
# those SVGs rather than guessed - see charts_design.md beside this notebook.
SURFACE = "#FFFFFF"        # figure and plot ground
GRID_COLOR = "#E4E7EB"     # a hairline grid the originals do not have; see below
AXIS_COLOR = "#444444"     # the axis line, exactly as the originals draw it
INK = "#000000"            # titles, tick labels, values
INK_MUTED = "#5A5A5A"      # subtitles and units

# Bars are separated by a gap in the surface colour, so the gap reads as space
# rather than as a drawn keyline.
BAR_GAP = SURFACE

# The one deliberate deviation from the originals: they carry no gridlines, which
# makes a value in the middle of a tall panel hard to read off. This grid is set
# light enough to sit under the data rather than compete with it. Set
# GRID_COLOR = SURFACE to switch it off and match the published figures exactly.

FONT_FAMILY = "Times New Roman"
TICK_SIZE, LEGEND_SIZE, PANEL_SIZE, TITLE_SIZE = 16, 16, 16, 21

plt.rcParams.update({
    "font.family": FONT_FAMILY,
    "font.size": TICK_SIZE,
    "svg.fonttype": "none",         # text stays text: editable, and a smaller file
    "figure.dpi": 100,
    "savefig.dpi": 100,
    "figure.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": AXIS_COLOR,
    "text.color": INK,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "legend.labelcolor": INK,
})

# ---------------------------------------------------------------- the palettes
#
# Every palette below was run through the palette validator against white in
# light mode, and the numbers quoted are that script's, not an impression.
# Delta E is OKLab x100; the checks want >= 8 for colour-blind separation and
# >= 15 for normal vision.

# One colour per country, taken from the published figures, keyed by the name as
# it appears in the long file.
#
# Keying on the data name rather than the printed label is the one fix kept here.
# The published charts keyed theirs by label, so wherever a legend wrote a country
# without spaces - "SaudiArabia", "SyrianArabRepublic", "UnitedArabEmirates",
# "StateofPalestine" - the lookup missed and plotly's default sequence took over.
# That is why Saudi Arabia is olive on one published chart and blue on the next.
# Keyed on the data name, a country keeps its colour across every chart.
#
# These colours do not pass the validator, and the numbers are worth knowing:
# United Arab Emirates #999999 has zero chroma (it is grey, not a hue) and sits
# at Delta E 1.0 from Tunisia under a deutan simulation; Iraq and Egypt are 7.8
# apart under normal vision, against a floor of 15. Ten of the twenty-two fall
# below 3:1 against white. They are kept because they are what the compendium
# prints. HOUSE_ALTERNATIVE below is a measured replacement - same job, worst
# pair anywhere 5.9 instead of 1.0, every colour clearing chroma and contrast -
# if the set is ever allowed to change.
HOUSE_COUNTRY_COLORS = {
    "Algeria": "#FECB51",              "Bahrain": "#405A29",
    "Comoros Islands": "#CE1227",      "Djibouti": "#056A3A",
    "Egypt": "#5ABD41",                "Iraq": "#67AA67",
    "Jordan": "#CF4473",               "Kuwait": "#524073",
    "Lebanon": "#6495C4",              "Libya": "#872F7C",
    "Mauritania": "#D28536",           "Morocco": "#6F70D2",
    "Oman": "#6B3BC3",                 "Palestine": "#77362E",
    "Qatar": "#C487C1",                "Saudi Arabia": "#A48C55",
    "Somalia": "#F6941C",              "Sudan": "#D24631",
    "Syrian Arab Republic": "#A7AB39", "Tunisia": "#52AA9B",
    "United Arab Emirates": "#999999", "Yemen": "#D48682",
}

# Chosen to maximise the worst pair *anywhere* rather than only between legend
# neighbours - twenty-one lines cross, so any two can end up side by side.
# Measured on white: worst pair anywhere 5.9 (against the house set's 1.0),
# worst legend neighbours 15.4, every colour at or above 3.11:1 contrast and
# 0.106 chroma. Wanting 3:1 on white is what forces these darker than the
# published set; 5.9 is the ceiling for twenty-two colours under that constraint.
HOUSE_ALTERNATIVE = {
    "Algeria": "#81036A",              "Bahrain": "#0B6BCB",
    "Comoros Islands": "#8E0B4D",      "Djibouti": "#17A1A1",
    "Egypt": "#0B0B65",                "Iraq": "#8E4C0B",
    "Jordan": "#6DA117",               "Kuwait": "#1776A1",
    "Lebanon": "#650B0B",              "Libya": "#0B248E",
    "Mauritania": "#0B8E75",           "Morocco": "#812303",
    "Oman": "#728103",                 "Palestine": "#AD0077",
    "Qatar": "#0B9BCB",                "Saudi Arabia": "#0B5D8E",
    "Somalia": "#17A164",              "Sudan": "#0B83CB",
    "Syrian Arab Republic": "#0B0BCB", "Tunisia": "#650B3E",
    "United Arab Emirates": "#004CAD", "Yemen": "#C2294F",
}

COUNTRY_COLORS = HOUSE_COUNTRY_COLORS      # swap to HOUSE_ALTERNATIVE to separate them
FALLBACK_COLOR = "#7F7F7F"

# The long file's country names are not the labels the compendium prints.
DISPLAY_NAMES = {"Comoros Islands": "Comoros", "Palestine": "State of Palestine"}

# The compendium's categorical triad, read out of 1.3, 1.7 and the pyramids:
# navy, pink, amber. It separates well - Male/Female Delta E 42.8 colour-blind,
# 50.9 normal; the age bands 16.2 and 30.1.
#
# The amber sits at 1.96:1 against white, which the validator flags as needing
# relief - a visible label or a table view - rather than as a free choice. That
# relief is already drawn: stacked_shares() prints the number inside every
# segment wide enough to hold one, and the workbook puts the whole table on the
# sheet beneath the picture.
#
# The published life-expectancy chart used blue and red here instead, but those
# are plotly's defaults showing through the same label-lookup miss described
# above, not a house choice - so the triad is used for every sex encoding.
SEX_COLORS = {"Male": "#003F5C", "Female": "#FFA600", "Both sexes": "#BC5090"}
AGE_BAND_COLORS = {"<15 years": "#003F5C", "15-64 years": "#BC5090", "65+ years": "#FFA600"}
SEX_LINE_COLORS = {"Male": "#003F5C", "Female": "#FFA600", "Both sexes": "#BC5090"}

SINGLE_BAR_COLOR = "#19516C"   # the single-series bar colour of 1.11

AGE_GROUPS = {
    "<15 years": ["0-4 years", "5-9 years", "10-14 years"],
    "15-64 years": ["15-19 years", "20-24 years", "25-29 years", "30-34 years",
                    "35-39 years", "40-44 years", "45-49 years", "50-54 years",
                    "55-59 years", "60-64 years"],
    "65+ years": ["65-69 years", "70-74 years", "75+ years"],
}
PYRAMID_BANDS = [band for bands in AGE_GROUPS.values() for band in bands]

GCC = ["Bahrain", "Kuwait", "Oman", "Qatar", "Saudi Arabia", "United Arab Emirates"]

# A country-year whose Male + Female misses its own reported Both-sexes total by
# more than this is a questionnaire error rather than a fact about the
# population - see drop_contradictory_sexes().
SEX_TOTAL_TOLERANCE = 2.0

## Reading a chapter

Parsing, the total-slice rules, and the checks that keep a mistyped digit off a chart.

In [ ]:
"""
CELL: Reading a chapter, parsing values, and the checks that keep bad cells off
the charts.
"""

# Problems met while charting. Collected rather than only logged, so the run
# cell can write every one to a file beside the charts - the log scrolls, and
# these are findings about the questionnaires that outlive the run.
FINDINGS = []


def note(kind, chapter, detail, **fields):
    """Record one data problem, and log it at warning level."""
    FINDINGS.append({"kind": kind, "chapter": chapter, "detail": detail, **fields})
    where = " ".join(str(value) for value in fields.values() if value not in (None, ""))
    logger.warning(f"  {kind}: {where} - {detail}" if where else f"  {kind}: {detail}")


def to_number(value):
    """Parse one Value cell into a float, or None if there is no number in it.

    The same parser notebook 3 uses, and for the same reason: these files store
    figures as text more often than as numbers, and not consistently. ' 701 956 '
    uses spaces as thousand separators, some cells carry a non-breaking space,
    a few hold '-' for "no data". float() alone fails on all but the plainest.
    """
    if pd.isna(value):
        return None
    text = str(value).replace("\xa0", " ").replace(",", "").strip()
    text = re.sub(r"\s+", "", text)
    if text in ("", "-", "--", "..", "..."):
        return None
    try:
        return float(text)
    except ValueError:
        return None


def country_label(name):
    """The name as the compendium prints it."""
    return DISPLAY_NAMES.get(name, name)


def country_color(name):
    return COUNTRY_COLORS.get(name, FALLBACK_COLOR)


def discover_chapters():
    """Chapters with a long file in the chosen language."""
    suffix = f"_{LANGUAGE}.xlsx"
    names = sorted(path.name[: -len(suffix)] for path in LONG_FILES_PATH.glob(f"*{suffix}")
                   if not path.name.endswith(f"_{LANGUAGE}_questionnaires.xlsx"))
    return names


def chapters_to_process():
    if CHAPTERS:
        return list(CHAPTERS)
    found = discover_chapters()
    if not found:
        logger.warning(f"No <Chapter>_{LANGUAGE}.xlsx in {LONG_FILES_PATH} - run notebooks 1-3 first.")
    return found


def charts_folder(chapter):
    """population_charts, labor_charts, ... - created on demand."""
    folder = CHARTS_ROOT / f"{chapter.lower()}_charts"
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def load_chapter(chapter, language=LANGUAGE):
    """The long file as a frame, with Value parsed into a numeric 'number' column.

    Rows whose Value will not parse are named before they are dropped - a cell
    someone filled with something the pipeline cannot use is worth reporting,
    because it is invisible otherwise.
    """
    table = pd.read_excel(LONG_FILES_PATH / f"{chapter}_{language}.xlsx", engine="openpyxl")
    table["number"] = table["Value"].map(to_number)

    unusable = table[table["number"].isna() & table["Value"].notna()]
    for value, group in unusable.groupby(unusable["Value"].astype(str)):
        countries = sorted({str(c) for c in group["Country"].dropna().unique()})[:4]
        note("value is not a number", chapter,
             f"{len(group)} cell(s) hold {value!r}, which cannot be plotted",
             country=", ".join(countries))

    table = table[table["number"].notna()].copy()
    table["Year"] = table["Year"].astype(int)
    return table


# Every column that can carry a breakdown, with the labels that mean "all of
# them", best first. total_slice() walks this so an indicator is reduced to one
# figure per country and year whatever breakdowns it happens to carry.
#
# Age Group needs a list rather than a single label. The labour rates are filed
# against 15+, 15-24, 15-64 and 25+ - overlapping populations, with no "Age
# Total" anywhere - so "the whole thing" for an unemployment rate is the 15+
# row. Without this the rates reduce to nothing and lose their charts.
WHOLE_LABELS = {
    "Sex": ["Both sexes"],
    "Nationality": ["Nationality Total"],
    "Area": ["Area Total"],
    "Age Group": ["Age Total", "15+ years", "15-64 years", "5-17 years", "15-24 years"],
    "Marital status": ["Marital status Total"],
}

# The canonical one, for callers that need a single label to exclude.
TOTAL_LABELS = {column: labels[0] for column, labels in WHOLE_LABELS.items()}


def total_slice(table, columns=None, keep=()):
    """Reduce to the total row on every breakdown column except those in `keep`.

    A breakdown is only filtered when its total row actually exists; an
    indicator that reports Nationals and Non-nationals but no total is left
    alone rather than emptied.
    """
    part = table
    for column, labels in WHOLE_LABELS.items():
        if column in keep or column not in part.columns:
            continue
        whole = next((label for label in labels if (part[column] == label).any()), None)
        if whole is not None:
            part = part[part[column] == whole]
        elif part[column].notna().any():
            # Only the parts are reported, never the whole. For a rate they
            # cannot be combined into a country figure without the populations
            # to weight them by, so they are left out rather than quietly
            # averaged into a number nobody reported.
            part = part[part[column].isna()]
    if columns:
        part = part[[c for c in columns if c in part.columns]]
    return part


def preferred_edition(table, indicators, chapter, keep=()):
    """Rows for an indicator that comes in a 'by nationality' and a 'by area' edition.

    The two describe the same people, so they are never concatenated - a country
    present in both would be counted twice. The first indicator listed wins, and
    the second only fills in countries the first does not cover. This is the
    rule notebook 3 already applies to the population counts; it is repeated
    here because fertility, infant mortality and the rest are split the same way
    and each edition covers a slightly different set of countries.
    """
    if isinstance(indicators, str):
        indicators = [indicators]

    frames = []
    for indicator in indicators:
        part = table[table["Indicator"] == indicator]
        if part.empty:
            continue
        part = total_slice(part, keep=keep).copy()
        part["source_indicator"] = indicator
        frames.append(part)

    if not frames:
        note("indicator missing", chapter, f"no usable values for {indicators[0]!r}")
        return pd.DataFrame(columns=list(table.columns) + ["source_indicator"])

    combined = pd.concat(frames, ignore_index=True)
    preferred = frames[0]["source_indicator"].iloc[0]
    covered = set(combined.loc[combined["source_indicator"] == preferred, "Country"])
    keep_rows = (combined["source_indicator"] == preferred) | (~combined["Country"].isin(covered))
    return combined[keep_rows]


# A breakdown whose parts are mutually exclusive and, between them, cover
# everybody. Only these may be added up to stand in for a missing total: Urban
# plus Rural is the whole country, and Nationals plus Non-nationals is everyone
# in it. "Nationals" on its own is not - see total_or_derived().
EXHAUSTIVE_PARTS = {
    "Area": ("Area Total", ["Urban", "Rural"]),
    "Nationality": ("Nationality Total", ["Nationals", "Non-nationals"]),
}

KEY_COLUMNS = ["Country", "Sex", "Age Group", "Year"]


def total_or_derived(part, column, chapter, indicator):
    """The total slice, with the total added up where the file never states one.

    Egypt is the case that forced this. It reports its population split Urban
    and Rural, and again split Nationals and Non-nationals, but files no total
    row for either - so a rule that takes only the total slice drops the largest
    country in the region without a word. Urban plus Rural is the whole country,
    so the total can be recovered exactly, and summing it back is not the
    double-counting the rule exists to prevent: that would be adding the
    nationality edition *to* the area edition, which still never happens.

    A partition is only summed when every one of its parts is present for that
    country, sex, age band and year. A country reporting Nationals and nothing
    else is left alone - its parts do not add up to a country.
    """
    total_label, parts = EXHAUSTIVE_PARTS[column]
    stated = part[part[column] == total_label]

    pieces = part[part[column].isin(parts)]
    if pieces.empty:
        return stated
    wide = pieces.pivot_table(index=KEY_COLUMNS, columns=column, values="number", aggfunc="sum")
    complete = wide.dropna(subset=[p for p in parts if p in wide.columns])
    if len(complete.columns) < len(parts) or complete.empty:
        return stated

    derived = complete.sum(axis=1).rename("number").reset_index()
    have = set(map(tuple, stated[KEY_COLUMNS].itertuples(index=False, name=None)))
    missing = derived[[tuple(row) not in have
                       for row in derived[KEY_COLUMNS].itertuples(index=False, name=None)]]
    if missing.empty:
        return stated

    for country in sorted({str(c) for c in missing["Country"].unique()}):
        rows = len(missing[missing["Country"] == country])
        note("total added up from its parts", chapter,
             f"{indicator}: no {total_label!r} row, so {rows} figure(s) were summed "
             f"from {' + '.join(parts)}", country=country)

    missing = missing.assign(**{column: total_label, "Indicator": indicator})
    return pd.concat([stated, missing], ignore_index=True)


def drop_scale_outliers(frame, chapter, indicator, factor=20):
    """Drop points that sit orders of magnitude off their own country's series.

    Morocco's 2024 population is filed as 36,491 - the country reported it in
    thousands - and Lebanon's 2022 as 100, which is a percentage that landed in
    a counts row. Neither is a population. A national population does not move
    twenty-fold in a year, so anything that far from the country's own median is
    a unit or a typing error rather than a fact, and plotting it flattens every
    real series on the chart to a straight line at the bottom.

    Only ever applied to counts of people. Rates genuinely can jump - a refugee
    population can multiply in a year - so those are left alone.
    """
    if frame.empty:
        return frame
    median = frame.groupby("Country")["number"].transform("median")
    ratio = frame["number"] / median.replace(0, pd.NA)
    suspect = ratio.notna() & ((ratio > factor) | (ratio < 1 / factor))
    for row in frame[suspect].itertuples():
        note("figure is off the scale of its own series", chapter,
             f"{indicator}: {row.number:,.0f} against a country median of "
             f"{median[row.Index]:,.0f} - looks like a unit or typing error, point dropped",
             country=row.Country, year=int(row.Year))
    return frame[~suspect]


def population_base(table, chapter):
    """Population counts as country / year / sex / age band, nothing counted twice.

    Same construction as notebook 3: the by-nationality and by-area indicators
    describe the same people, so only the total slice of each is taken and the
    two are never added together. What is new here is that a missing total is
    recovered from an exhaustive split rather than dropping the country.
    """
    frames = []
    for indicator, column, total in [
        ("Population size by nationality", "Nationality", "Nationality Total"),
        ("Population size by area", "Area", "Area Total"),
    ]:
        if column not in table.columns:
            continue
        part = table[table["Indicator"] == indicator]
        part = total_or_derived(part, column, chapter, indicator).copy()
        part["source_indicator"] = indicator
        frames.append(part)

    if not frames:
        note("indicator missing", chapter, "no population counts to chart")
        return pd.DataFrame(columns=["Country", "Sex", "Age Group", "Year", "number"])

    combined = pd.concat(frames, ignore_index=True)
    preferred = "Population size by nationality"
    covered = set(combined.loc[combined["source_indicator"] == preferred, "Country"])
    keep = (combined["source_indicator"] == preferred) | (~combined["Country"].isin(covered))
    base = combined[keep][KEY_COLUMNS + ["number"]]

    # Guard the all-ages totals, which is where a unit slip does the damage. The
    # age bands are left as they are: a band legitimately differs from its
    # country's median band by more than the factor.
    totals = base[base["Age Group"] == "Age Total"]
    kept = drop_scale_outliers(totals, chapter, "Population size")
    kept = drop_contradictory_sexes(kept, chapter)
    return pd.concat([base[base["Age Group"] != "Age Total"], kept], ignore_index=True)


def drop_contradictory_sexes(totals, chapter):
    """Drop country-years whose men and women do not add up to their own total.

    Kuwait 2020 files 2,743,617 men and 1,720,904 women against a total of
    464,521 - the total lost its leading digit. Tunisia 2015 has the opposite:
    a sound total of 11,162,700 against 55,662,100 men. Both are one mistyped
    figure, but nothing in the file says which of the two is the sound one, and
    guessing would put an invented number on a chart. So the whole country-year
    goes, and the finding goes to the file for whoever can ask the country.
    """
    if totals.empty:
        return totals
    wide = totals.pivot_table(index=["Country", "Year"], columns="Sex",
                              values="number", aggfunc="first")
    if not {"Male", "Female", "Both sexes"}.issubset(wide.columns):
        return totals

    checkable = wide.dropna(subset=["Male", "Female", "Both sexes"])
    if checkable.empty:
        return totals
    deviation = ((checkable["Male"] + checkable["Female"] - checkable["Both sexes"]).abs()
                 / checkable["Both sexes"].replace(0, pd.NA) * 100)
    contradictory = set(checkable.index[deviation.fillna(999) > SEX_TOTAL_TOLERANCE])
    for country, year in sorted(contradictory):
        row = wide.loc[(country, year)]
        note("sex figures contradict the reported total", chapter,
             f"male {row['Male']:,.0f} + female {row['Female']:,.0f} against a reported "
             f"total of {row['Both sexes']:,.0f} - the country-year is left off every chart",
             country=country, year=int(year))

    pairs = list(zip(totals["Country"], totals["Year"]))
    return totals[[pair not in contradictory for pair in pairs]]


def sexes_wide(base, chapter, age_group="Age Total"):
    """Male / Female / Both sexes side by side per country-year.

    No checking here - population_base() has already thrown out the country-years
    whose figures contradict each other, so anything reaching this point can be
    divided one by the other.
    """
    wide = base[base["Age Group"] == age_group].pivot_table(
        index=["Country", "Year"], columns="Sex", values="number", aggfunc="first")
    for needed in ("Male", "Female", "Both sexes"):
        if needed not in wide.columns:
            wide[needed] = pd.NA
    return wide

## The chart primitives

Five shapes - lines by country, small multiples, stacked shares, ranked bars, a pyramid. Every chart in the set is one of these. Sizes are worked out from the text that has to fit, which is why nothing clips.

In [ ]:
"""
CELL: The chart primitives - five shapes every chart in the set is built from.
"""

# The published charts carry no title: the caption lives in the Word document
# beside them. That works in the report and badly in a folder of 40 files, so a
# title is drawn by default and this switches it off for print.
DRAW_TITLES = True

LINE_WIDTH = 1.45          # 2 px at 100 dpi, as published
MARKER_SIZE = 5.0
BAR_EDGE = {"edgecolor": BAR_GAP, "linewidth": 0.9}   # the gap between fills, in the surface

# What every chart written this run shows, so a folder of SVGs is readable
# without opening them.
INDEX = []

# The numbers behind each chart, kept so the workbook can put a chart's own data
# on the sheet beside it. Captured in the primitives rather than in the twelve
# chart functions: the primitive is the last place that still holds exactly what
# was drawn, after every filter and guard has run, so the sheet cannot drift from
# the picture.
CHART_DATA = {}


def record_data(stem, frame):
    """Keep the tidy data a primitive just drew, indexed by chart."""
    if frame is not None and not frame.empty:
        CHART_DATA[stem] = frame.reset_index(drop=True)


def register(chapter, stem, title, subtitle, source_note=""):
    INDEX.append({"chapter": chapter, "file": stem, "title": title,
                  "shows": subtitle, "note": source_note})


def save(fig, chapter, stem, title="", subtitle="", source_note=""):
    """Write one figure to the chapter's folder in every configured format."""
    folder = charts_folder(chapter)
    for extension in FORMATS:
        fig.savefig(folder / f"{stem}.{extension}", format=extension)
    plt.close(fig)
    register(chapter, stem, title, subtitle, source_note)
    return stem


def style_axes(ax, grid_axis="y"):
    """Recessive frame: no box, one hairline grid, ticks that do not shout."""
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(AXIS_COLOR)
    ax.grid(axis=grid_axis, color=GRID_COLOR, linewidth=1.0, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(length=0, labelsize=TICK_SIZE, colors=INK)


def compact_number(value, _pos=None):
    """13,382,962 -> 13M. Axis ticks want the scale, not the digits.

    The digits are still on the chart: ranked_bars() prints the exact figure at
    the end of every bar. Spelling both out collided six labels into one smear.
    """
    for limit, suffix in ((1e9, "B"), (1e6, "M"), (1e3, "k")):
        if abs(value) >= limit:
            return f"{value / limit:g}{suffix}"
    return f"{value:g}"


def year_ticks(ax, years, step=None):
    """Whole years on the x axis, thinned so the labels never collide."""
    years = sorted({int(y) for y in years})
    if not years:
        return
    step = step or max(1, round(len(years) / 8))
    ax.set_xticks(years[::step])
    ax.set_xticklabels([str(y) for y in years[::step]])


def head_px(title, subtitle, width=58):
    """How tall the title block will be, before the figure has a height.

    Needed because the figure's height depends on the head, the bars and the
    legend together, and a fraction cannot be resolved until the height is
    fixed.
    """
    if not (DRAW_TITLES and title):
        return 16
    title_lines = len(textwrap.wrap(str(title), width) or [str(title)])
    subtitle_lines = len(textwrap.wrap(str(subtitle), int(width * 1.25))) if subtitle else 0
    return 16 + title_lines * round(TITLE_SIZE * 1.35) + subtitle_lines * round(PANEL_SIZE * 1.30) + 12


def draw_head(fig, title, subtitle, width=58):
    """Draw the title and subtitle, and return the figure fraction they used.

    Every caller lays its axes out below whatever this returns, so a long title
    pushes the plot down instead of landing on top of it.
    """
    if not (DRAW_TITLES and title):
        return 0.02
    height_px = fig.get_figheight() * 100
    title_step = TITLE_SIZE * 1.35 / height_px
    subtitle_step = PANEL_SIZE * 1.30 / height_px

    y = 0.985
    for line in textwrap.wrap(str(title), width) or [str(title)]:
        fig.text(0.02, y, line, ha="left", va="top", fontsize=TITLE_SIZE, color=INK)
        y -= title_step
    if subtitle:
        y -= 0.004
        for line in textwrap.wrap(str(subtitle), int(width * 1.25)) or [str(subtitle)]:
            fig.text(0.02, y, line, ha="left", va="top", fontsize=PANEL_SIZE, color=INK_MUTED)
            y -= subtitle_step
    return (0.985 - y) + 0.02


def widest_label_px(fig, labels, size):
    """Width in pixels of the longest label, drawn and measured."""
    renderer = fig.canvas.get_renderer()
    widest = 0.0
    for label in labels:
        probe = fig.text(0, 0, str(label), fontsize=size)
        widest = max(widest, probe.get_window_extent(renderer).width)
        probe.remove()
    return widest


def label_margin(fig, labels, size=TICK_SIZE, pad_px=24):
    """Left margin wide enough for the longest row label, as a figure fraction.

    The labels are drawn once and measured rather than estimated from their
    character count. Guessing at an average character width clipped
    "United Arab Emirates (2010)" by a third: in a proportional face the count
    says very little about the width.
    """
    widest = widest_label_px(fig, labels, size)
    return min(0.55, max(0.12, (widest + pad_px) / (fig.get_figwidth() * 100)))


def country_lines(series_by_country, chapter, stem, title, subtitle,
                  y_formatter=None, source_note=""):
    """One line per country, legend down the right - the shape of 1.1, 1.2, 1.6.

    Twenty-one lines is more than colour alone can separate, and the published
    charts have the same problem; the legend is what makes a given country
    findable, so it is always drawn and always in the same colours. Where a
    country's own path matters more than the ranking, prefer small_multiples().
    """
    fig = plt.figure(figsize=(8, 8))
    head = draw_head(fig, title, subtitle)
    # The plot stops well short of the published 0.73 so the longest label in
    # the set - "Syrian Arab Republic" - fits whole. The published SVGs cut it off.
    ax = fig.add_axes([0.10, 0.09, 0.52, 1 - head - 0.09])

    all_years = set()
    for country in sorted(series_by_country, key=country_label):
        points = series_by_country[country].dropna().sort_index()
        if points.empty:
            continue
        all_years.update(points.index)
        ax.plot(points.index, points.values, color=country_color(country),
                linewidth=LINE_WIDTH, marker="o", markersize=MARKER_SIZE,
                markeredgewidth=0, label=country_label(country), zorder=3)
    if not all_years:
        plt.close(fig)
        note("nothing to chart", chapter, f"{stem}: no country has usable values")
        return None

    style_axes(ax)
    year_ticks(ax, all_years)
    if y_formatter:
        ax.yaxis.set_major_formatter(y_formatter)
    ax.legend(loc="upper left", bbox_to_anchor=(1.03, 1.0), frameon=False,
              fontsize=LEGEND_SIZE, handlelength=1.3, handletextpad=0.5,
              borderaxespad=0, labelspacing=0.42, labelcolor=INK)

    drawn = {country_label(c): s.dropna().sort_index()
             for c, s in series_by_country.items() if not s.dropna().empty}
    record_data(stem, pd.DataFrame(drawn).sort_index().rename_axis("Year").reset_index())
    return save(fig, chapter, stem, title, subtitle, source_note)


def small_multiples(panels, chapter, stem, title, subtitle, series_colors,
                    ncols=3, y_formatter=None, source_note=""):
    """A panel per country on one shared y scale - the shape of 1.8, 1.9, 1.10.

    `panels` maps country -> {series name: Series indexed by year}. The y scale
    is shared across every panel on purpose: on separate scales a country whose
    fertility moved from 2.0 to 2.2 looks exactly like one that moved from 2 to 6.
    """
    panels = {c: s for c, s in panels.items() if any(not v.dropna().empty for v in s.values())}
    if not panels:
        note("nothing to chart", chapter, f"{stem}: no country has usable values")
        return None

    countries = sorted(panels, key=country_label)
    nrows = -(-len(countries) // ncols)
    fig = plt.figure(figsize=(8, max(4.2, 1.15 * nrows + 1.6)))
    head = draw_head(fig, title, subtitle)

    axes = fig.subplots(nrows, ncols, sharex=True, sharey=True, squeeze=False)
    # A shared legend needs a band of its own, or it lands on the first row of
    # panel titles.
    legend_gap = 0.055 if len(next(iter(panels.values()))) > 1 else 0.0
    fig.subplots_adjust(left=0.11, right=0.98, top=1 - head - 0.045 - legend_gap,
                        bottom=0.10, hspace=0.75, wspace=0.16)

    all_years = set()
    for index, country in enumerate(countries):
        ax = axes[index // ncols][index % ncols]
        for name, points in panels[country].items():
            points = points.dropna().sort_index()
            if points.empty:
                continue
            all_years.update(points.index)
            ax.plot(points.index, points.values,
                    color=series_colors.get(name, FALLBACK_COLOR),
                    linewidth=LINE_WIDTH, marker="o", markersize=3.6,
                    markeredgewidth=0, label=name, zorder=3)
        style_axes(ax)
        ax.set_title(country_label(country), fontsize=PANEL_SIZE, color=INK, pad=5)

    for index in range(len(countries), nrows * ncols):     # unused cells
        axes[index // ncols][index % ncols].set_visible(False)

    for row in axes:
        for ax in row:
            if not ax.get_visible():
                continue
            year_ticks(ax, all_years, step=max(1, round(len(all_years) / 3)))
            for label in ax.get_xticklabels():
                label.set_rotation(90)
            if y_formatter:
                ax.yaxis.set_major_formatter(y_formatter)

    # Shared x hides the tick labels on every panel but the bottom row, and the
    # bottom row is usually part empty - which left two of the three columns
    # with no years at all. The lowest panel that exists in each column gets them.
    for column in range(ncols):
        occupied = [row for row in range(nrows) if axes[row][column].get_visible()]
        if occupied:
            axes[occupied[-1]][column].tick_params(labelbottom=True)

    handles, labels = axes[0][0].get_legend_handles_labels()
    if len(labels) > 1:
        fig.legend(handles, labels, loc="upper right", bbox_to_anchor=(0.99, 1 - head + 0.008),
                   frameon=False, fontsize=LEGEND_SIZE, handlelength=1.3,
                   ncol=len(labels), labelcolor=INK)

    tidy = [{"Country": country_label(country), "Year": year, "Series": name, "Value": value}
            for country in countries
            for name, values in panels[country].items()
            for year, value in values.dropna().sort_index().items()]
    frame = pd.DataFrame(tidy)
    if not frame.empty:
        frame = frame.pivot_table(index=["Country", "Year"], columns="Series",
                                  values="Value").reset_index()
        frame.columns.name = None
    record_data(stem, frame)
    return save(fig, chapter, stem, title, subtitle, source_note)


def readable_on(hex_color):
    """Whichever of the ink and the surface can actually be read on that fill.

    Decided by measuring contrast rather than by a fixed lightness rule, because
    a fixed rule is only right for one theme and silently inverts on the other:
    on this light theme the pale amber needs the black ink and the navy needs the
    white surface, and on a dark ground it is the other way round.
    """
    def luminance(color):
        channels = []
        for i in (1, 3, 5):
            channel = int(color[i:i + 2], 16) / 255
            channels.append(channel / 12.92 if channel <= 0.04045
                            else ((channel + 0.055) / 1.055) ** 2.4)
        return 0.2126 * channels[0] + 0.7152 * channels[1] + 0.0722 * channels[2]

    fill = luminance(hex_color)

    def contrast_with(candidate):
        high, low = sorted((fill, luminance(candidate)), reverse=True)
        return (high + 0.05) / (low + 0.05)

    return max((INK, SURFACE), key=contrast_with)


def stacked_shares(panels, chapter, stem, title, subtitle, category_colors,
                   categories, source_note="", label_floor=7.0):
    """Percentages that sum to 100, stacked across one or two panels - 1.3 and 1.7.

    Every segment wide enough to hold one carries its own number. That is not
    decoration: the palette validator flags the house amber as falling under 3:1
    against white and asks for visible labels or a table view in return, and
    this is the label.
    """
    panels = {name: frame for name, frame in panels.items() if not frame.empty}
    if not panels:
        note("nothing to chart", chapter, f"{stem}: no country has a full breakdown")
        return None

    names = list(panels)
    rows = sorted(set().union(*(set(frame.index) for frame in panels.values())),
                  key=country_label)

    # Everything is measured first and the figure sized to fit, rather than the
    # other way round. A fixed height with a long legend left the bars a
    # sliver; a fixed height with a short one left a blank half-page.
    fig = plt.figure(figsize=(8, 6))
    width_px = fig.get_figwidth() * 100
    left = label_margin(fig, [country_label(r) for r in rows])
    slot_px = widest_label_px(fig, categories, LEGEND_SIZE) + 62
    legend_cols = max(1, min(len(categories), int((width_px - 40) // slot_px)))
    legend_rows = -(-len(categories) // legend_cols)

    head = head_px(title, subtitle)
    panel_titles_px = 30 if len(names) > 1 else 6
    floor_px = 44 + legend_rows * 32
    fig.set_figheight((head + panel_titles_px + 46 * len(rows) + floor_px) / 100)

    axes = fig.subplots(1, len(names), sharey=True, squeeze=False)[0]
    head_fraction = draw_head(fig, title, subtitle)
    fig.subplots_adjust(left=left, right=0.98,
                        top=1 - head_fraction - panel_titles_px / (fig.get_figheight() * 100),
                        bottom=floor_px / (fig.get_figheight() * 100), wspace=0.22)

    positions = list(range(len(rows)))
    for ax, name in zip(axes, names):
        frame = panels[name].reindex(rows)
        start = pd.Series(0.0, index=rows)
        for category in categories:
            if category not in frame.columns:
                continue
            widths = frame[category].fillna(0.0)
            color = category_colors.get(category, FALLBACK_COLOR)
            ax.barh(positions, widths.values, left=start.values, height=0.68,
                    color=color, label=category, zorder=3, **BAR_EDGE)
            for y, (width, base_x) in enumerate(zip(widths.values, start.values)):
                if width >= label_floor:
                    ax.text(base_x + width / 2, y, f"{width:.0f}", ha="center", va="center",
                            fontsize=13, color=readable_on(color), zorder=4)
            start = start + widths

        style_axes(ax, grid_axis="x")
        ax.set_xlim(0, 100)
        ax.set_xticks([0, 50, 100])
        ax.set_yticks(positions)
        ax.set_yticklabels([country_label(r) for r in rows])
        if len(names) > 1:
            ax.set_title(name, fontsize=PANEL_SIZE, color=INK, pad=7)

    # Shared y: inverting once flips every panel. Inverting each in turn would
    # flip it back on the second.
    axes[0].set_ylim(len(rows) - 0.6, -0.6)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.008),
               frameon=False, fontsize=LEGEND_SIZE, ncol=legend_cols,
               handlelength=1.4, labelcolor=INK, columnspacing=1.6)

    blocks = []
    for name in names:
        block = panels[name].reindex(rows)
        block = block[[c for c in categories if c in block.columns]].copy()
        block.insert(0, "Country", [country_label(r) for r in rows])
        if len(names) > 1:
            block.insert(0, "Panel", name)
        blocks.append(block.reset_index(drop=True))
    record_data(stem, pd.concat(blocks, ignore_index=True))
    return save(fig, chapter, stem, title, subtitle, source_note)


def ranked_bars(values, chapter, stem, title, subtitle, value_format="{:,.0f}",
                color=SINGLE_BAR_COLOR, source_note=""):
    """One bar per country, longest at the top - the shape of 1.11.

    A single series, so there is no legend: the title names it, and every bar
    carries its own figure.
    """
    values = values.dropna().sort_values(ascending=True)
    if values.empty:
        note("nothing to chart", chapter, f"{stem}: no country has a usable value")
        return None

    fig = plt.figure(figsize=(8, max(3.0, 0.55 * len(values) + 2.0)))
    head = draw_head(fig, title, subtitle)
    left = label_margin(fig, [country_label(i) for i in values.index])
    ax = fig.add_axes([left, 0.13, 0.96 - left, 1 - head - 0.13])

    ax.barh([country_label(i) for i in values.index], values.values, height=0.62,
            color=color, zorder=3, **BAR_EDGE)
    for y, value in enumerate(values.values):
        ax.text(value + values.max() * 0.015, y, value_format.format(value),
                va="center", ha="left", fontsize=15, color=INK, zorder=4)

    style_axes(ax, grid_axis="x")
    ax.set_xlim(0, values.max() * 1.22)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(4))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(compact_number))

    record_data(stem, pd.DataFrame({"Country": [country_label(i) for i in values.index],
                                    "Value": values.values})[::-1])
    return save(fig, chapter, stem, title, subtitle, source_note)


def pyramid(shares, chapter, stem, title, subtitle, source_note=""):
    """Male left, female right, both on one shared scale.

    The shared scale is the whole point of the shape - the two halves are only
    comparable if a centimetre means the same thing on each side.
    """
    fig = plt.figure(figsize=(7, 7))
    head = draw_head(fig, title, subtitle, width=48)
    axes = fig.subplots(1, 2, sharey=True, squeeze=False)[0]
    fig.subplots_adjust(left=0.15, right=0.97, top=1 - head - 0.05, bottom=0.09, wspace=0.26)

    positions = list(range(len(PYRAMID_BANDS)))
    limit = float(pd.DataFrame(shares).max().max()) * 1.12
    for ax, sex in zip(axes, ("Male", "Female")):
        ax.barh(positions, shares[sex].reindex(PYRAMID_BANDS).values, height=0.68,
                color=SEX_COLORS[sex], zorder=3, **BAR_EDGE)
        style_axes(ax, grid_axis="x")
        ax.set_xlim(0, limit)
        ax.set_title(sex, fontsize=PANEL_SIZE + 2, color=INK, pad=7)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
    axes[0].invert_xaxis()                       # only the left half mirrors

    axes[0].set_yticks(positions)
    axes[0].set_yticklabels([band.replace(" years", "") for band in PYRAMID_BANDS])
    axes[0].tick_params(labelleft=True)

    # Columns in the order the halves are drawn, not the order the dict happens
    # to hold them, so the sheet reads the same way round as the picture above it.
    record_data(stem, pd.DataFrame(shares).reindex(PYRAMID_BANDS)[["Male", "Female"]]
                .rename_axis("Age group").reset_index())
    return save(fig, chapter, stem, title, subtitle, source_note)

## The Population set

In [ ]:
"""
CELL: The Population chapter's own charts - the numbered set the compendium prints.
"""

# Filenames follow the published set (1.1_pop_growth, 1.2_pop_size, ...). 1.4
# and 1.5 are absent here because no such chart was supplied to copy; the
# numbering is left with those gaps rather than closed up, so the files still
# line up with the figure numbers in the report.


def snapshot_label(country, year):
    """A country label that carries the year, for charts showing one year only.

    Coverage is uneven - Saudi Arabia's latest full breakdown is 2024 and
    Kuwait's is 2025 - so a chart that silently mixes them is misleading. The
    year rides along in the label.
    """
    return f"{country_label(country)} ({int(year)})"


def latest_full_year(frame, by, required):
    """The most recent year in which `by` has every one of `required` present."""
    complete = [year for year, group in frame.groupby("Year")
                if required.issubset(set(group[by]))]
    return max(complete) if complete else None


def as_series_by_country(frame, value="number"):
    """country -> Series of value indexed by year.

    Grouped rather than indexed directly: two rows for one country-year would
    otherwise put two points at the same x and draw the line back on itself.
    """
    return {country: group.groupby("Year")[value].mean().sort_index()
            for country, group in frame.groupby("Country")}


def chart_population_growth(table, chapter):
    frame = preferred_edition(table, "Average annual population growth rate (%)", chapter)
    if frame.empty:
        return None
    return country_lines(
        as_series_by_country(frame), chapter, "1.1_pop_growth",
        "Average annual population growth rate", "Per cent per year",
        source_note="Average annual population growth rate (%)")


def chart_population_size(table, chapter, base):
    totals = base[(base["Sex"] == "Both sexes") & (base["Age Group"] == "Age Total")].copy()
    if totals.empty:
        note("nothing to chart", chapter, "1.2_pop_size: no all-ages totals")
        return None
    totals["millions"] = totals["number"] / 1e6
    return country_lines(
        as_series_by_country(totals, "millions"), chapter, "1.2_pop_size",
        "Population size", "Millions",
        source_note="Population size by nationality / by area, total slice only")


def chart_sex_composition_gcc(table, chapter):
    """Men and women as a share of nationals and of non-nationals, GCC.

    The split is what the chart is for: nationals sit near 50/50 everywhere,
    while the non-national population is three-quarters male in some of these
    countries. Two panels on one scale is what makes that gap legible.
    """
    frame = table[(table["Indicator"] == "Population size by nationality")
                  & (table["Age Group"] == "Age Total")
                  & (table["Sex"].isin(["Male", "Female"]))
                  & (table["Nationality"].isin(["Nationals", "Non-nationals"]))
                  & (table["Country"].isin(GCC))]
    if frame.empty:
        note("nothing to chart", chapter, "1.3_sex_comp_gcc: no national/non-national split")
        return None

    panels = {"National": {}, "Non-national": {}}
    for country, group in frame.groupby("Country"):
        year = latest_full_year(group, "Sex", {"Male", "Female"})
        if year is None:
            continue
        year_rows = group[group["Year"] == year]
        for nationality, panel in [("Nationals", "National"), ("Non-nationals", "Non-national")]:
            part = year_rows[year_rows["Nationality"] == nationality]
            wide = part.set_index("Sex")["number"]
            if not {"Male", "Female"}.issubset(wide.index) or wide.sum() <= 0:
                continue
            total = wide["Male"] + wide["Female"]
            panels[panel][snapshot_label(country, year)] = {
                "Male": 100 * wide["Male"] / total, "Female": 100 * wide["Female"] / total}

    panels = {name: pd.DataFrame(rows).T for name, rows in panels.items() if rows}
    if not panels:
        return None
    return stacked_shares(
        panels, chapter, "1.3_sex_comp_gcc",
        "Sex composition of the national and non-national population",
        "Per cent, latest year available for each country",
        SEX_COLORS, ["Male", "Female"],
        source_note="Population size by nationality, all ages")


def chart_sex_ratio(table, chapter, base):
    wide = sexes_wide(base, chapter)
    usable = wide.dropna(subset=["Male", "Female"])
    usable = usable[usable["Female"] > 0]
    if usable.empty:
        note("nothing to chart", chapter, "1.6_sex_ratio: no country has both sexes")
        return None
    ratio = (100 * usable["Male"] / usable["Female"]).rename("ratio").reset_index()
    return country_lines(
        as_series_by_country(ratio, "ratio"), chapter, "1.6_sex_ratio",
        "Sex ratio", "Men per 100 women, all ages",
        source_note="Population size by nationality / by area, total slice only")


def chart_age_and_sex(table, chapter, base):
    """The share of each sex falling in each of the three broad age bands."""
    band_of = {age: band for band, ages in AGE_GROUPS.items() for age in ages}
    frame = base[base["Sex"].isin(["Male", "Female"])].copy()
    frame["band"] = frame["Age Group"].map(band_of)
    frame = frame[frame["band"].notna()]
    if frame.empty:
        note("nothing to chart", chapter, "1.7_pop_age_sex: no five-year age bands")
        return None

    panels = {"Female": {}, "Male": {}}
    for country, group in frame.groupby("Country"):
        # One year per country, not one per sex. Only a year with all sixteen
        # bands can be turned into shares - a partial year would put the missing
        # band's people into the others - and it has to be the same year for
        # both sexes, or the country appears twice with half a row each.
        complete = [year for year, rows in group.groupby("Year")
                    if all(set(rows[rows["Sex"] == sex]["Age Group"]) == set(PYRAMID_BANDS)
                           for sex in ("Male", "Female"))]
        if not complete:
            note("age bands incomplete", chapter,
                 "no single year carries all sixteen five-year bands for both sexes - "
                 "left off 1.7_pop_age_sex", country=country)
            continue
        year = max(complete)
        for sex, sex_rows in group[group["Year"] == year].groupby("Sex"):
            totals = sex_rows.groupby("band")["number"].sum()
            if totals.sum() > 0:
                panels[sex][snapshot_label(country, year)] = 100 * totals / totals.sum()

    panels = {name: pd.DataFrame(rows).T for name, rows in panels.items() if rows}
    if not panels:
        return None
    return stacked_shares(
        panels, chapter, "1.7_pop_age_sex",
        "Population by broad age group and sex",
        "Per cent of each sex, latest year available for each country",
        AGE_BAND_COLORS, list(AGE_GROUPS), label_floor=6.0,
        source_note="Population size by nationality / by area, five-year age bands")


def chart_fertility(table, chapter):
    frame = preferred_edition(
        table, ["Total fertility rate (children per woman) by nationality",
                "Total fertility rate (children per woman) by area"], chapter)
    if frame.empty:
        return None
    panels = {country: {"Total fertility rate": group.set_index("Year")["number"].sort_index()}
              for country, group in frame.groupby("Country")}
    return small_multiples(
        panels, chapter, "1.8_fertility", "Total fertility rate",
        "Children per woman", {"Total fertility rate": SINGLE_BAR_COLOR},
        source_note="Total fertility rate by nationality, filled in from the by-area edition")


def chart_life_expectancy(table, chapter):
    frame = table[table["Indicator"] == "Life expectancy at birth (years)"]
    frame = total_slice(frame, keep={"Sex"})
    frame = frame[frame["Sex"].isin(["Male", "Female"])]
    if frame.empty:
        note("indicator missing", chapter, "no life expectancy by sex")
        return None
    panels = {}
    for country, group in frame.groupby("Country"):
        panels[country] = {sex: part.set_index("Year")["number"].sort_index()
                           for sex, part in group.groupby("Sex")}
    return small_multiples(
        panels, chapter, "1.9_life_exp", "Life expectancy at birth",
        "Years", SEX_LINE_COLORS,
        source_note="Life expectancy at birth (years), by sex")


def chart_infant_mortality(table, chapter):
    frame = preferred_edition(
        table, ["Infant mortality rate (per 1,000 livebirths) by nationality",
                "Infant mortality rate (per 1,000 livebirths) by area"], chapter)
    if frame.empty:
        return None
    panels = {country: {"Infant mortality rate": group.set_index("Year")["number"].sort_index()}
              for country, group in frame.groupby("Country")}
    return small_multiples(
        panels, chapter, "1.10_infant_mort", "Infant mortality rate",
        "Deaths under one year per 1,000 livebirths",
        {"Infant mortality rate": SINGLE_BAR_COLOR},
        source_note="Infant mortality rate by nationality, filled in from the by-area edition")


def chart_migrant_stock_gcc(table, chapter):
    frame = total_slice(table[table["Indicator"] == "International migrant stock (number)"])
    frame = frame[frame["Country"].isin(GCC)]
    if frame.empty:
        note("nothing to chart", chapter, "1.11_intl_migrant_gcc: no GCC migrant stock")
        return None
    latest = frame.sort_values("Year").groupby("Country").tail(1)
    values = pd.Series(latest["number"].values,
                       index=[snapshot_label(c, y) for c, y in zip(latest["Country"], latest["Year"])])
    return ranked_bars(
        values, chapter, "1.11_intl_migrant_gcc", "International migrant stock, GCC",
        "Number of people, latest year available for each country",
        source_note="International migrant stock (number), both sexes")


def chart_migrant_stock(table, chapter):
    frame = total_slice(table[table["Indicator"] == "International migrant stock (number)"]).copy()
    if frame.empty:
        note("indicator missing", chapter, "no international migrant stock")
        return None
    frame["millions"] = frame["number"] / 1e6
    return country_lines(
        as_series_by_country(frame, "millions"), chapter, "1.12_intl_migrant",
        "International migrant stock", "Millions of people",
        source_note="International migrant stock (number), both sexes")


def chart_refugees(table, chapter):
    frame = total_slice(table[table["Indicator"] == "Refugee population (number)"]).copy()
    if frame.empty:
        note("indicator missing", chapter, "no refugee population")
        return None
    frame["millions"] = frame["number"] / 1e6
    return country_lines(
        as_series_by_country(frame, "millions"), chapter, "1.13_refugees",
        "Refugee population", "Millions of people",
        source_note="Refugee population (number)")


def chart_pyramids(table, chapter, base):
    """One pyramid per country, in its latest year with a complete age breakdown.

    Bars are the share of the whole population, not of each sex - that is what
    the published Bahrain and Egypt pyramids plot, and it is what lets the two
    halves be read against each other.
    """
    frame = base[base["Sex"].isin(["Male", "Female"]) & base["Age Group"].isin(PYRAMID_BANDS)]
    written = []
    for country, group in frame.groupby("Country"):
        complete = [year for year, year_rows in group.groupby("Year")
                    if all(set(year_rows[year_rows["Sex"] == sex]["Age Group"]) == set(PYRAMID_BANDS)
                           for sex in ("Male", "Female"))]
        if not complete:
            note("age bands incomplete", chapter,
                 "no year carries all sixteen bands for both sexes - no pyramid drawn",
                 country=country)
            continue
        year = max(complete)
        year_rows = group[group["Year"] == year]
        total = year_rows["number"].sum()
        shares = (year_rows.pivot_table(index="Age Group", columns="Sex",
                                        values="number", aggfunc="first") / total * 100)
        stem = f"pyramid_{country_label(country).lower().replace(' ', '_')}"
        written.append(pyramid(
            shares, chapter, stem, f"Population pyramid - {country_label(country)}",
            f"Per cent of the total population, {int(year)}",
            source_note="Population size by nationality / by area, five-year age bands"))
    return written


def build_population_charts(table, chapter):
    """Every Population chart, each one isolated so a bad indicator cannot end the run."""
    base = population_base(table, chapter)
    jobs = [
        ("1.1 population growth", lambda: chart_population_growth(table, chapter)),
        ("1.2 population size", lambda: chart_population_size(table, chapter, base)),
        ("1.3 sex composition, GCC", lambda: chart_sex_composition_gcc(table, chapter)),
        ("1.6 sex ratio", lambda: chart_sex_ratio(table, chapter, base)),
        ("1.7 age and sex", lambda: chart_age_and_sex(table, chapter, base)),
        ("1.8 fertility", lambda: chart_fertility(table, chapter)),
        ("1.9 life expectancy", lambda: chart_life_expectancy(table, chapter)),
        ("1.10 infant mortality", lambda: chart_infant_mortality(table, chapter)),
        ("1.11 migrant stock, GCC", lambda: chart_migrant_stock_gcc(table, chapter)),
        ("1.12 migrant stock", lambda: chart_migrant_stock(table, chapter)),
        ("1.13 refugees", lambda: chart_refugees(table, chapter)),
        ("population pyramids", lambda: chart_pyramids(table, chapter, base)),
    ]

    written = []
    for step, job in jobs:
        try:
            result = job()
        except Exception as error:                    # one bad chart, not a dead run
            note("chart failed", chapter, f"{step}: {type(error).__name__}: {error}")
            continue
        written.extend(result if isinstance(result, list) else [result])
    return [stem for stem in written if stem]

## Every other chapter

In [ ]:
"""
CELL: The same treatment for every other chapter, driven off the data.
"""

# Population has a hand-written set because the compendium prints specific,
# numbered figures for it. Every other chapter gets these three shapes, chosen
# per indicator by what the indicator actually carries:
#
#   <n>_<indicator>_trend        one line per country, total slice
#   <n>_<indicator>_by_sex       a panel per country, men against women
#   <n>_<indicator>_<breakdown>  stacked shares in the latest year
#
# Nothing is drawn from a shape the data cannot support, so a chapter with no
# sex split simply has no by_sex chart rather than an empty one.

# The eight dark steps, validated as a group against this surface: every check
# passes outright - worst adjacent colour-blind Delta E 8.4, normal vision 19.3,
# and all eight clear 3:1 against the ground.
# The house triad first, so a two- or three-part breakdown comes out in the
# compendium's own colours, then five more measured against them. All-pairs on
# white: colour-blind Delta E 9.2, normal vision 16.6 - both above their gates.
CATEGORY_COLORS = ["#003F5C", "#BC5090", "#FFA600", "#8E2C0B",
                   "#2C8E0B", "#2283C3", "#0B45CB", "#810B8E"]

# Eight is the end of the validated order, and a ninth hue would have to be
# invented. The eighth slot is spent on "Other" instead.
MAX_CATEGORIES = 8

# Columns that can carry a composition. Checked in this order; the first one
# with something to show wins, so an indicator broken down by both occupation
# and nationality is charted by occupation.
BREAKDOWN_COLUMNS = [
    "Main occupation", "Economic activity", "Institutional sector",
    "Employment status", "Reasons for inactivity", "Causes of death",
    "Marital status", "Education level", "Educational sector", "Quintile",
    "Type of living quarter", "Tenure of housing unit", "Source of water supply",
    "Source of Lighting", "Types of sewage disposal system",
    "Nationality", "Area",
]

# Age Group is deliberately not on that list. The labour rates use overlapping
# bands - 15+ contains 15-24 - so stacking them to 100 per cent would invent a
# composition that does not exist. Where age bands really do partition (the
# population counts) the Population chapter charts them itself.

MIN_COUNTRIES = 3      # below this a chart says less than a sentence would
MIN_POINTS = 2         # a line needs two points


def slug(text, limit=42):
    """A filename-safe stub of an indicator name."""
    text = re.sub(r"\(.*?\)", " ", str(text))          # units belong in the subtitle
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()
    return text[:limit].strip("_")


def category_colors_for(categories):
    return {name: CATEGORY_COLORS[i % len(CATEGORY_COLORS)] for i, name in enumerate(categories)}


def fold_to_top(frame, limit=MAX_CATEGORIES):
    """Keep the largest categories and gather the rest into 'Other'.

    Past the eighth the validated colour order runs out, and inventing a ninth
    hue is the thing the palette rules forbid. Folding keeps the chart honest -
    the small categories are still in the total, just not separately coloured.
    """
    if frame.shape[1] <= limit:
        return frame, list(frame.columns)
    ranked = frame.mean(axis=0).sort_values(ascending=False)
    keep = list(ranked.index[: limit - 1])
    folded = frame[keep].copy()
    folded["Other"] = frame.drop(columns=keep).sum(axis=1)
    return folded, keep + ["Other"]


def chart_indicator_trend(frame, chapter, indicator, number):
    """One line per country, on the indicator's total slice."""
    totals = total_slice(frame)
    counts = totals.groupby("Country")["Year"].nunique()
    countries = counts[counts >= MIN_POINTS].index
    totals = totals[totals["Country"].isin(countries)]
    if totals["Country"].nunique() < MIN_COUNTRIES:
        return None
    series = {country: group.groupby("Year")["number"].mean().sort_index()
              for country, group in totals.groupby("Country")}
    return country_lines(series, chapter, f"{number}_{slug(indicator)}_trend",
                         str(indicator), "All countries with a reported total",
                         source_note=str(indicator))


def chart_indicator_by_sex(frame, chapter, indicator, number):
    """A panel per country, men against women."""
    by_sex = total_slice(frame, keep={"Sex"})
    by_sex = by_sex[by_sex["Sex"].isin(["Male", "Female"])]
    if by_sex.empty:
        return None
    panels = {}
    for country, group in by_sex.groupby("Country"):
        sexes = {sex: part.groupby("Year")["number"].mean().sort_index()
                 for sex, part in group.groupby("Sex")}
        if len(sexes) == 2 and max(len(s) for s in sexes.values()) >= MIN_POINTS:
            panels[country] = sexes
    if len(panels) < MIN_COUNTRIES:
        return None
    return small_multiples(panels, chapter, f"{number}_{slug(indicator)}_by_sex",
                           str(indicator), "Men and women, by country",
                           SEX_LINE_COLORS, source_note=str(indicator))


def chart_indicator_composition(frame, chapter, indicator, number):
    """Stacked shares of whichever breakdown the indicator carries."""
    for column in BREAKDOWN_COLUMNS:
        if column not in frame.columns:
            continue
        total_label = TOTAL_LABELS.get(column)
        part = total_slice(frame, keep={column})
        part = part[part[column].notna()]
        if total_label:
            part = part[part[column] != total_label]
        categories = sorted({str(v) for v in part[column].unique()})
        if len(categories) < 2:
            continue

        rows = {}
        for country, group in part.groupby("Country"):
            year = latest_full_year(group, column, set(categories))
            if year is None:
                continue
            shares = group[group["Year"] == year].groupby(column)["number"].sum()
            if shares.sum() <= 0:
                continue
            rows[snapshot_label(country, year)] = 100 * shares / shares.sum()
        if len(rows) < MIN_COUNTRIES:
            continue

        table_ = pd.DataFrame(rows).T
        table_, order = fold_to_top(table_)
        return stacked_shares(
            {str(column): table_}, chapter, f"{number}_{slug(indicator)}_{slug(column, 24)}",
            str(indicator), f"Per cent by {str(column).lower()}, latest year available",
            category_colors_for(order), order, source_note=str(indicator))
    return None


def build_generic_charts(table, chapter):
    """Trend, by-sex and composition charts for every indicator that supports them."""
    written = []
    indicators = sorted({str(i) for i in table["Indicator"].dropna().unique()})
    for position, indicator in enumerate(indicators, start=1):
        frame = table[table["Indicator"].astype(str) == indicator]
        number = f"{position:02d}"
        for step, job in [
            ("trend", chart_indicator_trend),
            ("by sex", chart_indicator_by_sex),
            ("composition", chart_indicator_composition),
        ]:
            try:
                result = job(frame, chapter, indicator, number)
            except Exception as error:            # one bad chart, not a dead run
                note("chart failed", chapter,
                     f"{indicator} ({step}): {type(error).__name__}: {error}")
                continue
            if result:
                written.append(result)
    if not written:
        note("nothing to chart", chapter, "no indicator carried enough data for any chart shape")
    return written


def build_chapter_charts(chapter):
    """Population gets its published set; everything else gets the data-driven one."""
    table = load_chapter(chapter)
    logger.info(f"  {chapter}: {len(table):,} usable rows, "
                f"{table['Indicator'].nunique()} indicator(s), "
                f"{table['Country'].nunique()} countries")
    if chapter.lower() == "population":
        return build_population_charts(table, chapter)
    return build_generic_charts(table, chapter)

## The workbook

One file per chapter: a sheet per chart, holding that chart's picture with the rows that produced it underneath.

In [ ]:
"""
CELL: One workbook per chapter - a sheet per chart, holding that chart's own
data with the picture above it.

The point is that a figure and the numbers behind it stop being two artefacts
that can drift apart. Whatever the guards in the reading cell dropped is dropped
from the sheet too, because both come from the same capture inside the primitive
that drew the marks.

Excel places images through Pillow, which does not rasterise SVG, so the picture
embedded in a sheet is the PNG. The SVG is written to the same folder and is the
one to use anywhere that wants vector - it keeps its text as text.
"""
from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

WORKBOOK_SUFFIX = "_charts.xlsx"
ROW_PIXELS = 20                      # one default-height Excel row
FORBIDDEN_IN_SHEET_NAME = set("[]:*?/") | {chr(92)}

SHEET_FILL = PatternFill("solid", fgColor=SURFACE.lstrip("#"))
HEADER_FILL = PatternFill("solid", fgColor=GRID_COLOR.lstrip("#"))
INK_FONT = Font(name=FONT_FAMILY, size=11, color=INK.lstrip("#"))
MUTED_FONT = Font(name=FONT_FAMILY, size=10, color=INK_MUTED.lstrip("#"))
HEADER_FONT = Font(name=FONT_FAMILY, size=11, bold=True, color=INK.lstrip("#"))
TITLE_FONT = Font(name=FONT_FAMILY, size=15, bold=True, color=INK.lstrip("#"))


def sheet_name(stem, taken):
    r"""Excel allows 31 characters, forbids [ ] : * ? / \, and wants them unique."""
    clean = "".join("-" if c in FORBIDDEN_IN_SHEET_NAME else c for c in stem)[:31]
    name, n = clean, 2
    while name.lower() in taken:
        suffix = f"~{n}"
        name, n = clean[:31 - len(suffix)] + suffix, n + 1
    taken.add(name.lower())
    return name


def cell_value(value):
    """numpy scalars and NaN do not survive openpyxl; hand it Python or nothing."""
    if value is None:
        return None
    if not isinstance(value, str):
        try:
            if pd.isna(value):
                return None
        except (TypeError, ValueError):
            pass                      # anything array-like is not a missing value
    return value.item() if hasattr(value, "item") else value


def paint(worksheet, last_row, last_column):
    """The chart surface under the whole used range, so the picture sits flush."""
    worksheet.sheet_view.showGridLines = False
    for row in worksheet.iter_rows(min_row=1, max_row=last_row + 4,
                                   min_col=1, max_col=max(last_column + 2, 14)):
        for cell in row:
            cell.fill = SHEET_FILL


def write_sheet(workbook, chapter, entry, frame, taken):
    """One chart: its heading, its picture, then the numbers that produced it."""
    worksheet = workbook.create_sheet(sheet_name(entry["file"], taken))
    worksheet.sheet_properties.tabColor = SINGLE_BAR_COLOR.lstrip("#")

    worksheet["A1"] = entry["title"] or entry["file"]
    worksheet["A1"].font = TITLE_FONT
    worksheet["A2"] = entry["shows"] or ""
    worksheet["A2"].font = MUTED_FONT
    note_bits = [b for b in (entry.get("note"), f"{entry['file']}.svg") if b]
    worksheet["A3"] = "  |  ".join(note_bits)
    worksheet["A3"].font = MUTED_FONT

    image_rows = 0
    png = charts_folder(chapter) / f"{entry['file']}.png"
    if png.exists():
        picture = XLImage(png)
        worksheet.add_image(picture, "A5")
        image_rows = -(-picture.height // ROW_PIXELS)
    else:
        note("chart image missing", chapter, f"{entry['file']}.png was not found")

    top = 5 + image_rows + 2
    for column, name in enumerate(frame.columns, start=1):
        cell = worksheet.cell(row=top, column=column, value=str(name))
        cell.font, cell.fill = HEADER_FONT, HEADER_FILL
        cell.alignment = Alignment(horizontal="center")
    for r, (_, record) in enumerate(frame.iterrows(), start=top + 1):
        for c, name in enumerate(frame.columns, start=1):
            cell = worksheet.cell(row=r, column=c, value=cell_value(record[name]))
            cell.font = INK_FONT
            if isinstance(cell.value, float):
                cell.number_format = "0.0#"

    for column, name in enumerate(frame.columns, start=1):
        longest = max([len(str(name))] + [len(f"{v}") for v in frame[name].head(200)])
        worksheet.column_dimensions[get_column_letter(column)].width = min(28, max(10, longest + 3))
    paint(worksheet, top + len(frame), len(frame.columns))
    return worksheet.title


def write_index_sheet(workbook, chapter, placed):
    """A contents page - every chart, the sheet holding it, and what it shows."""
    worksheet = workbook.create_sheet("Index", 0)
    worksheet["A1"] = f"{chapter} - charts and the data behind them"
    worksheet["A1"].font = TITLE_FONT
    worksheet["A2"] = (f"{len(placed)} charts. Each sheet carries one chart's picture and "
                       "exactly the rows that produced it.")
    worksheet["A2"].font = MUTED_FONT

    headers = ["Sheet", "Chart file", "Title", "Shows", "Rows of data"]
    for column, name in enumerate(headers, start=1):
        cell = worksheet.cell(row=4, column=column, value=name)
        cell.font, cell.fill = HEADER_FONT, HEADER_FILL
    for r, item in enumerate(placed, start=5):
        for c, value in enumerate([item["sheet"], f"{item['file']}.svg", item["title"],
                                   item["shows"], item["rows"]], start=1):
            cell = worksheet.cell(row=r, column=c, value=value)
            cell.font = INK_FONT
        worksheet.cell(row=r, column=1).hyperlink = f"#'{item['sheet']}'!A1"
        worksheet.cell(row=r, column=1).font = Font(
            name=FONT_FAMILY, size=11, color=SINGLE_BAR_COLOR.lstrip("#"), underline="single")
    for column, width in zip("ABCDE", (26, 30, 46, 60, 13)):
        worksheet.column_dimensions[column].width = width
    paint(worksheet, 4 + len(placed), 5)


def write_workbook(chapter):
    """Collect this chapter's charts into one file beside the SVGs."""
    entries = [row for row in INDEX if row["chapter"] == chapter]
    if not entries:
        return None

    workbook = Workbook()
    workbook.remove(workbook.active)
    taken, placed = set(), []
    for entry in entries:
        frame = CHART_DATA.get(entry["file"])
        if frame is None or frame.empty:
            note("no data captured", chapter,
                 f"{entry['file']} was drawn but its data was not captured - no sheet written")
            continue
        name = write_sheet(workbook, chapter, entry, frame, taken)
        placed.append({"sheet": name, "file": entry["file"], "title": entry["title"],
                       "shows": entry["shows"], "rows": len(frame)})

    if not placed:
        note("empty workbook", chapter, "no chart had data to write")
        return None
    write_index_sheet(workbook, chapter, placed)

    path = charts_folder(chapter) / f"{chapter.lower()}{WORKBOOK_SUFFIX}"
    workbook.save(path)
    logger.info("%s: workbook with %d chart sheets -> %s", chapter, len(placed), path.name)
    return path

## Run

In [ ]:
"""
CELL: Main run - every chart for every chapter, plus the index and the findings file.
"""


def write_index(chapter):
    """A one-line-per-chart index beside the SVGs, so the folder reads without opening."""
    rows = [row for row in INDEX if row["chapter"] == chapter]
    if not rows:
        return None
    path = charts_folder(chapter) / "charts_index.csv"
    pd.DataFrame(rows)[["file", "title", "shows", "note"]].to_csv(path, index=False,
                                                                 encoding="utf-8-sig")
    return path


def write_findings(chapter):
    """Everything the charts could not use, written where the charts are.

    These are findings about the questionnaires, not about the code - a mistyped
    digit belongs back with the country that reported it - so they outlive the
    run the same way notebook 3's inconsistencies do.
    """
    rows = [row for row in FINDINGS if row["chapter"] == chapter]
    path = charts_folder(chapter) / FINDINGS_NAME
    lines = [f"{chapter} - data problems met while charting",
             f"{len(rows)} finding(s)", "=" * 70, ""]
    for row in rows:
        where = ", ".join(f"{key}={value}" for key, value in row.items()
                          if key not in ("kind", "chapter", "detail") and value not in (None, ""))
        lines.append(f"[{row['kind']}] {where}" if where else f"[{row['kind']}]")
        lines.append(f"    {row['detail']}")
        lines.append("")
    path.write_text("\n".join(lines), encoding="utf-8")
    return path, len(rows)


print(f"Reading  {LONG_FILES_PATH}\\<Chapter>_{LANGUAGE}.xlsx")
print(f"Writing  {CHARTS_ROOT}\\<chapter>_charts\\\n")

FINDINGS.clear()
INDEX.clear()
CHART_DATA.clear()

chapters = chapters_to_process()
built, finding_counts, workbooks = {}, {}, {}
steps, step = max(len(chapters), 1), 0

for chapter in chapters:
    step += 1
    bar = "#" * step + "-" * (steps - step)
    print(f"[{bar}] {step}/{steps}  {chapter}")
    try:
        built[chapter] = build_chapter_charts(chapter)
    except Exception as error:          # a bad chapter must not end the run
        note("chapter failed", chapter, f"{type(error).__name__}: {error}")
        built[chapter] = []
    write_index(chapter)
    _, finding_counts[chapter] = write_findings(chapter)
    # The workbook is written last: it reads INDEX and CHART_DATA, which are only
    # complete once every chart for the chapter has been drawn.
    try:
        workbooks[chapter] = write_workbook(chapter)
    except Exception as error:
        note("workbook failed", chapter, f"{type(error).__name__}: {error}")
        workbooks[chapter] = None

print("\n" + "=" * 70)
print("CHARTS WRITTEN")
print("=" * 70)
if not any(built.values()):
    print("None - run notebooks 1-3 first, or check the log above.")
else:
    for chapter, stems in sorted(built.items()):
        folder = f"{chapter.lower()}_charts"
        formats = "/".join(FORMATS)
        print(f"  {folder:<24} {len(stems):>3} chart(s) as {formats}"
              f"   {finding_counts.get(chapter, 0)} finding(s)")
    total = sum(len(stems) for stems in built.values())
    print(f"\n  {total} chart(s). Each folder also holds charts_index.csv (what every")
    print(f"  file shows) and {FINDINGS_NAME} (cells the charts could not use).")
    print("\n  Workbooks - one sheet per chart, its picture above its own data:")
    for chapter, path in sorted(workbooks.items()):
        if path is None:
            print(f"    {chapter:<14} not written - see {FINDINGS_NAME}")
        else:
            sheets = sum(1 for row in INDEX
                         if row["chapter"] == chapter and row["file"] in CHART_DATA)
            print(f"    {chapter:<14} {path.name}  ({sheets} chart sheets + Index)")

## Verify

Re-opens what was written. Everything here is on OneDrive, where a write landing is not the same as a write staying.

In [ ]:
"""
CELL: Verify - re-open what was written and check it is a real chart.
"""
import openpyxl


def verify_charts():
    """Re-read every file this run produced and check it holds a drawn chart.

    Worth doing rather than trusting the run: matplotlib will write a perfectly
    valid SVG with nothing in it, and everything is on OneDrive, where a write
    landing is not the same as a write staying.
    """
    if not INDEX:
        print("Nothing to verify - run the cell above first.")
        return

    problems = []
    for chapter in sorted({row["chapter"] for row in INDEX}):
        folder = charts_folder(chapter)
        expected = [row["file"] for row in INDEX if row["chapter"] == chapter]
        print(f"\n{chapter}  ->  {folder}")

        for stem in expected:
            for extension in FORMATS:
                path = folder / f"{stem}.{extension}"
                if not path.exists():
                    problems.append(f"{chapter}: {path.name} was not written")
                    continue
                size = path.stat().st_size
                if size < 2000:
                    problems.append(f"{chapter}: {path.name} is only {size} bytes")
                if extension == "svg":
                    text = path.read_text(encoding="utf-8", errors="ignore")
                    if "<svg" not in text:
                        problems.append(f"{chapter}: {path.name} is not an SVG")
                    # A chart with no marks has no path or rect in its axes.
                    elif text.count("<path") + text.count("<rect") < 3:
                        problems.append(f"{chapter}: {path.name} has no drawn data")

        index_path = folder / "charts_index.csv"
        if not index_path.exists():
            problems.append(f"{chapter}: charts_index.csv is missing")
        else:
            listed = set(pd.read_csv(index_path)["file"])
            missing = sorted(set(expected) - listed)
            if missing:
                problems.append(f"{chapter}: index does not list {missing}")

        # The workbook is the deliverable now, so check it the same way: reopen
        # it and confirm every chart really has both a picture and its numbers.
        book_path = folder / f"{chapter.lower()}{WORKBOOK_SUFFIX}"
        if not book_path.exists():
            problems.append(f"{chapter}: {book_path.name} is missing")
        else:
            book = openpyxl.load_workbook(book_path)
            if "Index" not in book.sheetnames:
                problems.append(f"{chapter}: workbook has no Index sheet")
            charted = [row["file"] for row in INDEX
                       if row["chapter"] == chapter and row["file"] in CHART_DATA]
            for stem in charted:
                match = [n for n in book.sheetnames if n == stem[:31]]
                if not match:
                    problems.append(f"{chapter}: workbook has no sheet for {stem}")
                    continue
                sheet = book[match[0]]
                if not sheet._images:
                    problems.append(f"{chapter}: {match[0]} carries no picture")
                if sheet.max_row < 8 or sheet.max_column < 2:
                    problems.append(f"{chapter}: {match[0]} carries no data table")
            book.close()
            print(f"  workbook {book_path.name}: {len(book.sheetnames)} sheet(s), "
                  f"{book_path.stat().st_size / 1e6:.1f} MB")

        svgs = sorted(folder.glob("*.svg"))
        stale = sorted({p.stem for p in svgs} - set(expected))
        print(f"  {len(expected)} chart(s) checked, {len(svgs)} svg file(s) on disk")
        if stale:
            print(f"  left over from an earlier run: {', '.join(stale)}")

    print("\n" + "=" * 70)
    if problems:
        print(f"{len(problems)} PROBLEM(S)")
        for problem in problems:
            print(f"  - {problem}")
    else:
        print("Every chart was written, is a real SVG, and carries drawn data.")
    return problems


verify_charts()